<a href="https://colab.research.google.com/github/SATHRAMCHARAN/CSA6102---DIGITAL-FORENSICS-AND-CYBERCRIME-INVENTIGATION/blob/main/exp29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")

def detect_bruteforce(events, threshold=5, window_minutes=2):
    """
    Detect brute-force login attempts:
    - >= threshold failed logons (4625)
    - for the same account within window_minutes
    - report whether a successful logon (4624) followed.
    """
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))

    by_account = {}
    for e in events:
        by_account.setdefault(e["account"], []).append(e)

    results = {}

    for account, acc_events in by_account.items():
        failures = [e for e in acc_events if e["event_id"] == 4625]
        successes = [e for e in acc_events if e["event_id"] == 4624]

        flagged = False
        success_after = False

        for i in range(len(failures)):
            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)

            count = sum(
                1
                for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )

            if count >= threshold:
                flagged = True

                # Check if a successful login happened after the attack window
                for s in successes:
                    if parse_time(s["timestamp"]) > window_end:
                        success_after = True
                        break
                break

        results[account] = {
            "bruteforce_detected": flagged,
            "success_after": success_after,
        }

    return results